# Biomedical Image Analysis Self-Study Notebook

## Topic: 2D Cell and Nuclei Segmentation with a Pretrained Model

This notebook is designed for self-directed learning in biomedical image analysis. It follows a concept-first, code-second structure:

1. Understand the concept.
2. Run the matching code.
3. Complete a small task.
4. Reflect on the result.

The notebook focuses on **segmentation**, one of the most common tasks in biomedical image analysis. Segmentation assigns a label to each pixel so that structures such as nuclei, cells, lesions, or organs can be measured.

No training is required in this notebook. A pretrained Cellpose model is used for inference. Optional training-related code is included only as commented reference material near the end.

### Learning goals

By the end of this notebook, you should be able to:

- Explain what a segmentation mask represents.
- Prepare a biomedical image for model inference.
- Run a pretrained segmentation model.
- Visualize masks and object boundaries.
- Compute basic segmentation agreement metrics.
- Extract simple morphology features from segmented objects.
- Identify common failure modes in biomedical segmentation.


## 0. Environment setup

The code below installs and imports the packages used in this notebook.

Cellpose provides pretrained models for cell and nuclei segmentation. In this notebook, we use a pretrained model only for inference. The model may be downloaded automatically by Cellpose, or you can manually download a pretrained model file from the URL defined below.

All downloads are stored in the notebook directory:

- Example image: `./data/mitosis.tif`
- Pretrained models: `./pretrained_models/`

Pretrained model URL used in this notebook:

`https://huggingface.co/spaces/mouseland/cellpose/resolve/main/cyto3`

If you are running this notebook in Google Colab, uncomment and run the installation cell. If you already have the packages installed, you can skip it.


In [ ]:
# If needed, uncomment the next line to install dependencies.
# %pip install -q "cellpose==3.1.1.1" "scikit-image" "matplotlib" "numpy" "pandas" "opencv-python-headless"


In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import exposure, filters, measure, morphology, segmentation, util
from skimage.color import label2rgb

# Use the notebook directory for local data and model storage.
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
MODEL_DIR = NOTEBOOK_DIR / "pretrained_models"
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
os.environ["CELLPOSE_LOCAL_MODELS_PATH"] = str(MODEL_DIR)

print("Notebook directory:", NOTEBOOK_DIR)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)

# Cellpose is imported later with a helpful error message.

plt.rcParams["figure.figsize"] = (7, 7)
np.random.seed(42)


## 1. Concept: What is segmentation?

In image classification, one label is assigned to a whole image. In segmentation, one label is assigned to each pixel.

A simple binary segmentation task can be written as:

$$
f_\theta(x) \rightarrow \hat{y}, \quad \hat{y}_{i,j} \in \{0, 1\}
$$

where $x$ is the input image, $f_\theta$ is the model, and $\hat{y}$ is the predicted mask. A pixel value of $1$ may represent foreground, such as a nucleus, while $0$ represents background.

For instance segmentation, the mask stores object identities:

$$
\hat{y}_{i,j} \in \{0, 1, 2, \dots, N\}
$$

where $0$ is background and each positive integer represents a different object instance.

### Why segmentation matters in biomedical image analysis

Segmentation enables measurements such as:

- cell count
- nuclear area
- organ volume
- lesion burden
- spatial distribution of tissue structures

### Student task 1

Before running the code, write down one biomedical research question that would require segmentation rather than classification.


## 2. Code: Load an example biomedical image

We will use a small microscopy-style image. If `data/mitosis.tif` is not already present in the notebook directory, it will be downloaded there on first run.


In [ ]:
from skimage import io

MITOSIS_URL = (
    "https://gitlab.com/scikit-image/data/-/raw/"
    "2cdc5ce89b334d28f06a58c9f0ca21aa6992a5ba/"
    "AS_09125_050116030001_D03f00d0.tif"
)
MITOSIS_PATH = DATA_DIR / "mitosis.tif"

if not MITOSIS_PATH.exists():
    try:
        print("Downloading example image to:", MITOSIS_PATH)
        urllib.request.urlretrieve(MITOSIS_URL, MITOSIS_PATH)
        print("Download complete.")
    except Exception as e:
        raise FileNotFoundError(
            f"Could not download mitosis.tif to {MITOSIS_PATH}. "
            "Please download the file manually and rerun this cell."
        ) from e
else:
    print("Using local image:", MITOSIS_PATH)

image = io.imread(MITOSIS_PATH)

print("Image shape:", image.shape)
print("Data type:", image.dtype)
print("Minimum intensity:", image.min())
print("Maximum intensity:", image.max())

plt.imshow(image, cmap="gray")
plt.title("Input biomedical image")
plt.axis("off")
plt.show()


## 3. Concept: Intensity normalization

Biomedical images often come from different scanners, microscopes, staining protocols, or acquisition settings. Raw intensity values may not be directly comparable across images.

A common normalization approach rescales the image using robust percentiles:

$$
x_{norm} = \frac{x - p_{low}}{p_{high} - p_{low}}
$$

where $p_{low}$ and $p_{high}$ are lower and upper intensity percentiles, such as the 1st and 99th percentiles.

After clipping, the image is usually mapped into the range $[0, 1]$.

Normalization helps the model see a more consistent input distribution.


## 4. Code: Normalize and inspect the image

The code below rescales the image using robust percentiles and then plots both the image and its intensity histogram.


In [ ]:
def robust_normalize(img, low_percentile=1, high_percentile=99):
    """Normalize an image to [0, 1] using robust percentiles."""
    img = img.astype(np.float32)
    p_low, p_high = np.percentile(img, (low_percentile, high_percentile))
    img = np.clip(img, p_low, p_high)
    img = (img - p_low) / (p_high - p_low + 1e-8)
    return img

image_norm = robust_normalize(image)

plt.imshow(image_norm, cmap="gray")
plt.title("Robustly normalized image")
plt.axis("off")
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(image_norm.ravel(), bins=50)
plt.title("Intensity histogram after normalization")
plt.xlabel("Normalized intensity")
plt.ylabel("Pixel count")
plt.show()


### Student task 2

Change the percentile range in `robust_normalize`, for example from `(1, 99)` to `(5, 95)`. Observe how the visual contrast changes.

Questions:

1. Does stronger clipping make object boundaries easier or harder to see?
2. Could aggressive normalization remove meaningful biological signal?
3. How would you choose normalization settings for a real dataset?


In [ ]:
# TODO: Experiment with a different percentile range.
# image_norm_alt = robust_normalize(image, low_percentile=5, high_percentile=95)
# plt.imshow(image_norm_alt, cmap="gray")
# plt.title("Alternative normalization")
# plt.axis("off")
# plt.show()


## 5. Concept: Pretrained segmentation models

A pretrained model has already learned useful visual patterns from another dataset. For self-study, this is useful because we can focus on image representation, inference, visualization, and evaluation without spending time training.

This notebook uses Cellpose-style pretrained cell segmentation models. In practice, pretrained models should still be validated on your own data because biomedical images can vary strongly across:

- imaging modality
- staining protocol
- microscope type
- scanner type
- resolution
- tissue type
- disease state

A pretrained model is not automatically correct for a new biomedical domain.

### Important distinction

Inference uses fixed model parameters:

$$
\hat{y} = f_\theta(x)
$$

Training updates model parameters:

$$
\theta^* = \arg\min_\theta \mathcal{L}(f_\theta(x), y)
$$

This notebook performs inference only.


## 6. Code: Download or load a pretrained model

Cellpose can automatically load pretrained models by name. The optional code below also shows how to download a pretrained model file from a URL.

The manual download step is optional. If it fails, the notebook will still try to use Cellpose's built-in pretrained model loading.


### About the pretrained model

This notebook uses Cellpose's pretrained nuclei segmentation model for inference. A pretrained model is useful here because it can segment cell nuclei directly without training on a new dataset first.

The main inference cell below loads the built-in `nuclei` model. If you want to inspect the optional downloadable weights used by the notebook, the model file comes from:

- Model weights: https://huggingface.co/spaces/mouseland/cellpose/resolve/main/cyto3
- Cellpose model documentation: https://cellpose.readthedocs.io/en/latest/models.html

If you need a figure or example screenshot for your notes, the Cellpose model documentation page is the best place to start.

In [ ]:
PRETRAINED_MODEL_URL = "https://huggingface.co/spaces/mouseland/cellpose/resolve/main/cyto3"
MODEL_PATH = MODEL_DIR / "cyto3"

if not MODEL_PATH.exists():
    try:
        print("Downloading pretrained model file to:", MODEL_PATH)
        urllib.request.urlretrieve(PRETRAINED_MODEL_URL, MODEL_PATH)
        print("Download complete.")
    except Exception as e:
        print("Manual model download failed. Cellpose built-in model loading may still work.")
        print("Reason:", repr(e))
else:
    print("Model file already exists:", MODEL_PATH)


In [ ]:
try:
    from cellpose import models
    CELLPOSE_AVAILABLE = True
    print("Cellpose imported successfully.")
except Exception as e:
    CELLPOSE_AVAILABLE = False
    print("Cellpose is not available. Install it before running model inference.")
    print("Reason:", repr(e))


## 7. Concept: Inference parameters

Segmentation models often require a few inference settings. For Cellpose-style inference, useful parameters include:

- `model_type`: which pretrained model to use, such as nuclei or cytoplasm.
- `diameter`: expected object diameter in pixels. If set to `None`, the model estimates it.
- `channels`: how image channels should be interpreted.

For a grayscale image, a common channel setting is `[0, 0]`, meaning that the same single-channel image is used as input.

### Why object diameter matters

If objects are expected to have diameter $d$ pixels, the model can use this scale information to separate nearby instances. If $d$ is too small, one object may be split into fragments. If $d$ is too large, nearby objects may be merged.


## 8. Code: Run pretrained model inference

The following cell runs segmentation inference. It does not train a model.


In [ ]:
if not CELLPOSE_AVAILABLE:
    raise ImportError("Cellpose is not installed. Run the installation cell first.")

use_gpu = False

# Option A: load a built-in pretrained model by name.
# For nuclei-rich images, model_type="nuclei" is often a good first choice.
model = models.Cellpose(gpu=use_gpu, model_type="nuclei")

# Option B: if you want to load the manually downloaded model file, try this instead.
# model = models.CellposeModel(gpu=use_gpu, pretrained_model=str(MODEL_PATH))

masks, flows, styles, estimated_diameter = model.eval(
    image_norm,
    diameter=None,
    channels=[0, 0]
)

print("Estimated diameter:", estimated_diameter)
print("Mask shape:", masks.shape)
print("Number of segmented objects:", int(masks.max()))


## 9. Concept: Visualizing segmentation masks

A segmentation result should not be trusted only because it was produced by a model. Visual inspection is a critical first step.

Common visualization methods include:

- overlaying colored masks on the original image
- drawing object boundaries
- assigning each object a random color
- comparing predictions with manual annotations

A good visualization should help you answer:

1. Are objects detected?
2. Are nearby objects separated?
3. Are boundaries accurate?
4. Are there false positives in the background?
5. Are faint objects missed?


## 10. Code: Visualize masks and boundaries


In [ ]:
# Convert instance labels into an RGB overlay.
overlay = label2rgb(masks, image=image_norm, bg_label=0, alpha=0.35)

# Extract object boundaries.
boundaries = segmentation.find_boundaries(masks, mode="outer")

plt.imshow(image_norm, cmap="gray")
plt.imshow(np.ma.masked_where(~boundaries, boundaries), alpha=0.8)
plt.title("Predicted object boundaries")
plt.axis("off")
plt.show()

plt.imshow(overlay)
plt.title("Predicted instance masks overlaid on image")
plt.axis("off")
plt.show()


### Student task 3

Run inference with a fixed `diameter`, such as `diameter=15`, `diameter=30`, or `diameter=60`.

Questions:

1. Which diameter gives the most plausible separation of objects?
2. What types of errors appear when the diameter is too small?
3. What types of errors appear when the diameter is too large?


In [ ]:
# TODO: Try different diameter values.
# test_diameter = 30
# masks_diameter, flows_diameter, styles_diameter, estimated_diameter_diameter = model.eval(
#     image_norm,
#     diameter=test_diameter,
#     channels=[0, 0]
# )
# overlay_diameter = label2rgb(masks_diameter, image=image_norm, bg_label=0, alpha=0.35)
# plt.imshow(overlay_diameter)
# plt.title(f"Segmentation with fixed diameter = {test_diameter}")
# plt.axis("off")
# plt.show()
# print("Number of objects:", int(masks_diameter.max()))


## 11. Concept: Baselines and agreement metrics

A classical image processing baseline can be useful for comparison. For example, Otsu thresholding chooses a threshold that separates foreground and background based on the image histogram.

The result is not a true ground truth label. However, comparing a model prediction with a simple baseline can reveal whether the model behaves very differently from a basic intensity-based method.

Two common binary segmentation metrics are Dice and Intersection over Union.

Dice coefficient:

$$
\mathrm{Dice}(A, B) = \frac{2|A \cap B|}{|A| + |B|}
$$

Intersection over Union:

$$
\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

Here, $A$ and $B$ are binary masks. Both metrics range from $0$ to $1$, where $1$ means perfect overlap.

### Important warning

If $B$ is only a threshold baseline, these metrics measure agreement with the baseline, not accuracy against expert annotation.


## 12. Code: Create a threshold baseline and compute agreement


In [ ]:
# Create a simple Otsu threshold baseline.
threshold = filters.threshold_otsu(image_norm)
baseline_binary = image_norm > threshold
baseline_binary = morphology.remove_small_objects(baseline_binary, min_size=20)
baseline_binary = morphology.remove_small_holes(baseline_binary, area_threshold=20)

# Convert instance mask to binary foreground.
pred_binary = masks > 0

def dice_score(mask_a, mask_b):
    """Compute the Dice coefficient between two binary masks."""
    mask_a = mask_a.astype(bool)
    mask_b = mask_b.astype(bool)
    intersection = np.logical_and(mask_a, mask_b).sum()
    denominator = mask_a.sum() + mask_b.sum()
    return (2 * intersection) / (denominator + 1e-8)

def iou_score(mask_a, mask_b):
    """Compute the Intersection over Union between two binary masks."""
    mask_a = mask_a.astype(bool)
    mask_b = mask_b.astype(bool)
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return intersection / (union + 1e-8)

print("Otsu threshold:", threshold)
print("Dice agreement with threshold baseline:", dice_score(pred_binary, baseline_binary))
print("IoU agreement with threshold baseline:", iou_score(pred_binary, baseline_binary))

plt.imshow(baseline_binary, cmap="gray")
plt.title("Threshold baseline foreground mask")
plt.axis("off")
plt.show()


### Student task 4

Modify the baseline by changing `min_size` and `area_threshold`.

Questions:

1. How do small-object filtering settings affect Dice and IoU?
2. Does a higher Dice score always mean the segmentation is visually better?
3. Why is expert annotation needed for real validation?


In [ ]:
# TODO: Change these values and recompute metrics.
# min_size = 50
# area_threshold = 50
# baseline_modified = image_norm > threshold
# baseline_modified = morphology.remove_small_objects(baseline_modified, min_size=min_size)
# baseline_modified = morphology.remove_small_holes(baseline_modified, area_threshold=area_threshold)
# print("Modified Dice:", dice_score(pred_binary, baseline_modified))
# print("Modified IoU:", iou_score(pred_binary, baseline_modified))


## 13. Concept: Instance-level morphology features

After segmentation, each object can be measured. This is one reason segmentation is so important in biomedical image analysis.

For object $i$, area is the number of foreground pixels belonging to that object:

$$
A_i = \sum_{j,k} \mathbf{1}(\hat{y}_{j,k} = i)
$$

Other useful features include:

- centroid location
- bounding box
- eccentricity
- perimeter
- mean intensity
- major and minor axis length

These features can be used for quality control, downstream statistics, or biological interpretation.


## 14. Code: Extract morphology features


In [ ]:
features = measure.regionprops_table(
    masks,
    intensity_image=image_norm,
    properties=(
        "label",
        "area",
        "centroid",
        "eccentricity",
        "mean_intensity",
        "major_axis_length",
        "minor_axis_length",
    ),
)

features_df = pd.DataFrame(features)
features_df.head()


In [ ]:
print("Number of measured objects:", len(features_df))

if len(features_df) > 0:
    display(features_df.describe())

    plt.figure(figsize=(7, 4))
    plt.hist(features_df["area"], bins=30)
    plt.title("Distribution of object areas")
    plt.xlabel("Area in pixels")
    plt.ylabel("Object count")
    plt.show()


### Student task 5

Use the feature table to answer the following questions:

1. What is the median object area?
2. Are there very small objects that may be false positives?
3. Which object has the highest mean intensity?
4. How might morphology features help detect segmentation failures?


In [ ]:
# TODO: Explore the feature table.
# median_area = features_df["area"].median()
# smallest_objects = features_df.sort_values("area").head(10)
# brightest_object = features_df.sort_values("mean_intensity", ascending=False).head(1)
# display(median_area)
# display(smallest_objects)
# display(brightest_object)


## 15. Concept: Post-processing segmentation masks

Post-processing applies simple rules after model prediction. These rules are not learned by the model, but they can improve usability.

Common post-processing steps include:

- removing very small objects
- filling small holes
- smoothing boundaries
- separating touching objects
- excluding objects outside a region of interest

A simple area filter keeps only objects with sufficient area:

$$
\hat{y}' = \{i : A_i \geq A_{min}\}
$$

Post-processing should be used carefully. If rules are too aggressive, true biological structures may be removed.


## 16. Code: Remove very small predicted objects


In [ ]:
def filter_instances_by_area(label_image, min_area=50):
    """Remove labeled instances smaller than min_area and relabel the result."""
    filtered = np.zeros_like(label_image, dtype=np.int32)
    current_label = 1

    for region in measure.regionprops(label_image):
        if region.area >= min_area:
            filtered[label_image == region.label] = current_label
            current_label += 1

    return filtered

min_area = 50
masks_filtered = filter_instances_by_area(masks, min_area=min_area)

print("Objects before filtering:", int(masks.max()))
print("Objects after filtering:", int(masks_filtered.max()))

filtered_overlay = label2rgb(masks_filtered, image=image_norm, bg_label=0, alpha=0.35)
plt.imshow(filtered_overlay)
plt.title(f"Masks after area filtering, min_area = {min_area}")
plt.axis("off")
plt.show()


### Student task 6

Change `min_area` and observe the result.

Questions:

1. Which objects disappear first?
2. Are the removed objects more likely to be noise or true biological structures?
3. What evidence would you need before choosing a final `min_area` threshold?


In [ ]:
# TODO: Try a different area threshold.
# min_area_alt = 100
# masks_filtered_alt = filter_instances_by_area(masks, min_area=min_area_alt)
# overlay_alt = label2rgb(masks_filtered_alt, image=image_norm, bg_label=0, alpha=0.35)
# plt.imshow(overlay_alt)
# plt.title(f"Masks after area filtering, min_area = {min_area_alt}")
# plt.axis("off")
# plt.show()


## 17. Concept: Error analysis

Segmentation error analysis is the process of identifying when and why a model fails. This is essential in biomedical image analysis because model errors can bias downstream measurements.

Common error types include:

- false positive objects
- missed faint objects
- merged neighboring objects
- split single objects
- inaccurate boundaries
- sensitivity to intensity normalization
- sensitivity to object size assumptions

A useful error analysis table might include:

| Error type | Visual sign | Possible cause | Possible fix |
|---|---|---|---|
| Missed object | No mask on visible structure | Low contrast | Improve preprocessing or annotation |
| Merged objects | One mask covers several objects | Diameter too large | Adjust diameter or post-processing |
| Split object | One object becomes multiple masks | Diameter too small | Adjust diameter or smoothing |
| False positive | Mask on background | Noise or artifact | Area filtering or domain-specific QC |


## 18. Code: Create an error-review panel

This cell creates a compact visual panel that can be used for manual error analysis.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_norm, cmap="gray")
axes[0].set_title("Input image")
axes[0].axis("off")

axes[1].imshow(label2rgb(masks, image=image_norm, bg_label=0, alpha=0.35))
axes[1].set_title("Raw predicted masks")
axes[1].axis("off")

axes[2].imshow(label2rgb(masks_filtered, image=image_norm, bg_label=0, alpha=0.35))
axes[2].set_title("Filtered predicted masks")
axes[2].axis("off")

plt.show()


### Student task 7

Inspect the panel above and manually list at least five suspected segmentation errors.

For each suspected error, record:

1. The approximate location in the image.
2. The error type.
3. The likely cause.
4. One possible improvement.

This task is intentionally qualitative. Biomedical image analysis requires both computational and visual reasoning.


## 19. Optional extension: Use your own image

You can replace the example image with your own microscopy or biomedical image. The image should be a 2D grayscale or RGB image.

Recommended workflow:

1. Load the image.
2. Convert it to grayscale if needed.
3. Normalize intensities.
4. Run pretrained segmentation inference.
5. Visualize masks.
6. Perform error analysis.

The code below is a template. Uncomment and adapt it for your own file.


In [ ]:
# TODO: Use your own image file.
# from skimage import io, color
#
# image_path = DATA_DIR / "your_image.png"
# user_image = io.imread(image_path)
#
# if user_image.ndim == 3:
#     user_image_gray = color.rgb2gray(user_image)
# else:
#     user_image_gray = user_image
#
# user_image_norm = robust_normalize(user_image_gray)
#
# user_masks, user_flows, user_styles, user_estimated_diameter = model.eval(
#     user_image_norm,
#     diameter=None,
#     channels=[0, 0]
# )
#
# plt.imshow(label2rgb(user_masks, image=user_image_norm, bg_label=0, alpha=0.35))
# plt.title("Segmentation on your own image")
# plt.axis("off")
# plt.show()


## 20. Optional reference only: Training is intentionally not run

Training is outside the scope of this self-study notebook. It requires labeled data, careful validation, and usually GPU resources.

The following cell is intentionally commented out. Students who already have annotated data and understand the workflow may adapt it later.


In [ ]:
# This training skeleton is intentionally commented out.
# Do not run it as part of this notebook.
#
# from cellpose import train
#
# train_images = [...]  # List of training images
# train_masks = [...]   # List of expert annotation masks
#
# # Example only. Exact arguments depend on the Cellpose version and dataset format.
# # model_path, train_losses, test_losses = train.train_seg(
# #     net=model.net,
# #     train_data=train_images,
# #     train_labels=train_masks,
# #     channels=[0, 0],
# #     n_epochs=100,
# #     learning_rate=0.2,
# #     weight_decay=1e-5,
# # )


## 21. Summary

In this notebook, you learned how to:

- define pixel-wise and instance-wise segmentation
- normalize biomedical image intensities
- use a pretrained model for inference
- visualize segmentation masks and boundaries
- compare a prediction with a simple threshold baseline
- compute Dice and IoU agreement scores
- extract morphology features from segmented objects
- perform qualitative error analysis
- understand why training is separate from inference

### Suggested final reflection

Write a short paragraph answering this question:

**If this segmentation model were used in a real biomedical study, what validation evidence would you require before trusting the measurements?**


## References and further reading

- MIDeL: Medical Imaging & Deep Learning, Mayo Radiology Informatics Lab.  
  https://mayo-radiology-informatics-lab.github.io/MIDeL/index.html
- Cellpose documentation.  
  https://cellpose.readthedocs.io/
- Cellpose website.  
  https://www.cellpose.org/
- MONAI Model Zoo.  
  https://project-monai.github.io/model-zoo.html
- Bio-image Analysis Notebooks: Cellpose example.  
  https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/20b_deep_learning/cellpose.html
